# Modelo jerárquico con Binomial Negativa para touchdowns

## Qué responde este notebook

Los modelos de las etapas anteriores describen al jugador con promedios rezagados
(`_last3_avg`, `_season_avg`, `_career_avg`). Los datos son un panel: los mismos receptores repetidos
semana tras semana. Un modelo jerárquico (de efectos mixtos) modela esa estructura directamente: un
efecto por jugador captura «quién es este receptor» con toda su historia, encogido hacia el promedio
de la posición cuando hay poca información (*shrinkage*). Se combina con una verosimilitud Binomial
Negativa, la opción estándar para conteos con más varianza que una Poisson. En touchdowns la
varianza supera a la media en el agregado, pero según
[`4.1_preparacion_y_metricas.ipynb`](4.1_preparacion_y_metricas.ipynb) ese exceso viene de que la
tasa cambia entre receptores y partidos, lo que aquí captan las variables y el efecto por jugador;
el azar de cada partido es incluso menor que el de una Poisson. La Binomial Negativa puede entonces
no aportar nada sobre una Poisson; se prueba por ser la opción estándar.

La pregunta es si este modelo, más explicable, predice mejor que el XGBoost con objetivo Poisson
elegido en [`4.4_ajuste_hiperparametros.ipynb`](4.4_ajuste_hiperparametros.ipynb), con la regla de
siempre: deviance de Poisson en validación y mejora real según el intervalo bootstrap. Usa
`pymc`/`bambi`/`arviz` (Bayesiano), incluidos en `docker/requirements.txt`.

In [1]:
import sys
sys.path.insert(0, "../src")
import data, features, modeling, experimentos

import numpy as np
import pandas as pd
import bambi as bmb
import arviz as az
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor

pd.set_option("display.width", 160)

AZUL, NARANJA, VERDE, GRIS, GRIS_CLARO = "#2a78d6", "#eb6834", "#1baf7a", "#888780", "#c3c2b7"

wr = features.construir_tabla_modelado(range(2016, 2026))
features_ridge = features.columnas_por_modelo("ridge")
train_mask = wr["season"].between(2016, 2021)
val_mask = wr["season"].between(2022, 2023)
test_mask = wr["season"].between(2024, 2025)
print(f"features (set Ridge, ya sin colinealidad severa): {len(features_ridge)}")

CONFIG_ELEGIDA = {
    "receptions": dict(max_depth=3, n_estimators=200, learning_rate=0.03),
    "receiving_yards": dict(max_depth=3, n_estimators=200, learning_rate=0.03),
    "receiving_tds": dict(objective="count:poisson", max_depth=3, n_estimators=200, learning_rate=0.08),
}

features (set Ridge, ya sin colinealidad severa): 30


## 1. Antes de ajustar: qué perdería un `dropna()`

A diferencia de XGBoost, que maneja faltantes de forma nativa, un modelo de `bambi` necesita datos
completos. Antes de decidir cómo completarlos, conviene ver qué se perdería al descartar filas.

In [2]:
perdidas = wr.loc[train_mask, features_ridge].isna().any(axis=1)
print(f"filas de train que un dropna() perderia: {perdidas.sum()} de {train_mask.sum()} ({perdidas.mean():.1%})")

nulos_por_columna = wr.loc[train_mask, features_ridge].isna().sum().sort_values(ascending=False)
print("\ncolumnas con mas nulos:")
print(nulos_por_columna.head(6))

# caso concreto: una fila con nulos en _season_avg es literalmente la primera semana
# de esa temporada para ese jugador -- season_avg (shift(1).expanding()) no tiene nada
# anterior de donde promediar todavia.
ejemplo = wr.loc[train_mask & wr["receiving_tds_season_avg"].isna()].iloc[0]
print(f"\nejemplo real: {ejemplo['player_display_name']}, season={int(ejemplo['season'])} week={int(ejemplo['week'])}")
print(f"receiving_tds_season_avg={ejemplo['receiving_tds_season_avg']} (nulo -- es su primera semana jugada esa temporada)")

filas de train que un dropna() perderia: 2589 de 13795 (18.8%)

columnas con mas nulos:
receiving_tds_season_avg          1371
fantasy_points_ppr_season_avg     1371
receptions_season_avg             1371
receiving_yards_season_avg        1371
receiving_air_yards_season_avg    1371
targets_season_avg                1371
dtype: int64

ejemplo real: Steve Smith, season=2016 week=1
receiving_tds_season_avg=nan (nulo -- es su primera semana jugada esa temporada)


**Lectura.** Un `dropna()` perdería 2,589 de 13,795 filas de entrenamiento (18.8%). Las columnas con más nulos
son las `_season_avg` (1,371 cada una), que por construcción están vacías en la primera semana de
cada temporada: todavía no hay partidos de esa temporada que promediar. Descartarlas no quita
filas al azar, quita todos los inicios de temporada. Por eso se imputa: mediana para las variables
numéricas y valor más frecuente para la categoría de experiencia, ajustados solo con
entrenamiento.

In [3]:
cols_numericas = [c for c in features_ridge if c != "anios_experiencia_bucket"]
imputador_num = SimpleImputer(strategy="median")
imputador_cat = SimpleImputer(strategy="most_frequent")

def imputar(df, ajustar):
    fn_num = imputador_num.fit_transform if ajustar else imputador_num.transform
    fn_cat = imputador_cat.fit_transform if ajustar else imputador_cat.transform
    num = pd.DataFrame(fn_num(df[cols_numericas]), index=df.index, columns=cols_numericas)
    cat = pd.DataFrame(fn_cat(df[["anios_experiencia_bucket"]]), index=df.index, columns=["anios_experiencia_bucket"])
    return pd.concat([num, cat], axis=1)[features_ridge]

target = "receiving_tds"
X_train_imp = imputar(wr.loc[train_mask, features_ridge], ajustar=True)
datos_train = X_train_imp.copy()
datos_train[target] = wr.loc[train_mask, target].to_numpy()
datos_train["player_id"] = wr.loc[train_mask, "player_id"].astype(str).to_numpy()
print(f"filas de entrenamiento tras imputar: {len(datos_train)} (0 perdidas, vs. {perdidas.sum()} que se hubieran perdido)")

filas de entrenamiento tras imputar: 13795 (0 perdidas, vs. 2589 que se hubieran perdido)


## 2. El modelo: 30 variables + un efecto por jugador, verosimilitud Binomial Negativa

`receiving_tds ~ [30 variables de la versión lineal] + (1|player_id)`, en la sintaxis de fórmula
de `lme4`, que `bambi` traduce a un modelo de PyMC. Se usan los priors por defecto de `bambi`
(débilmente informativos, en la escala de los datos).

In [4]:
formula = f"{target} ~ " + " + ".join(features_ridge) + " + (1|player_id)"
modelo = bmb.Model(formula, datos_train, family="negativebinomial")

import time
t0 = time.time()
idata = modelo.fit(draws=500, tune=500, chains=4, cores=4, random_seed=0, progressbar=False)
print(f"tiempo de ajuste (500 muestras + 500 de calentamiento, 4 cadenas): {time.time()-t0:.1f}s")

Initializing NUTS using jitter+adapt_diag...


/usr/local/lib/python3.13/site-packages/pytensor/tensor/rewriting/elemwise.py:1004: UserWarning: Loop fusion failed because the resulting node would exceed the kernel argument limit.
  warn(


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [alpha, Intercept, anios_experiencia, catch_rate_career_avg, depth_team, draft_pick, edad, fantasy_points_ppr_last3_avg, fantasy_points_ppr_last5_avg, fantasy_points_ppr_season_avg, receiving_10_last3_avg, receiving_16_last5_avg, receiving_20_last3_avg, receiving_air_yards_season_avg, receiving_epa_last3_avg, receiving_first_downs_career_avg, receiving_first_downs_last5_avg, receiving_tds_season_avg, receiving_yards_career_avg, receiving_yards_last3_avg, receiving_yards_season_avg, receptions_career_avg, receptions_last3_avg, receptions_last5_avg, receptions_season_avg, targets_last3_avg, targets_last5_avg, targets_season_avg, wopr_last3_avg, wopr_last5_avg, anios_experiencia_sq, anios_experiencia_bucket, 1|player_id_sigma, 1|player_id_offset]


Sampling 4 chains for 500 tune and 500 draw iterations (2_000 + 2_000 draws total) took 571 seconds.


The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


tiempo de ajuste (500 muestras + 500 de calentamiento, 4 cadenas): 585.4s


## 3. Diagnóstico de convergencia sobre todos los parámetros

El modelo tiene un efecto por cada jugador de entrenamiento, más los coeficientes fijos, el
intercepto, la dispersión de la Binomial Negativa y la varianza del efecto de jugador. Se revisa
`r_hat` en todos; debería quedar por debajo de 1.01 (Vehtari et al., 2021).

In [5]:
resumen_completo = az.summary(idata)
problematicos = resumen_completo[resumen_completo["r_hat"] > 1.01]
es_efecto_jugador = problematicos.index.str.startswith("1|player_id[")
print(f"parametros totales: {len(resumen_completo)}")
print(f"r_hat maximo: {resumen_completo['r_hat'].max():.4f}")
print(f"parametros con r_hat > 1.01: {len(problematicos)} ({len(problematicos)/len(resumen_completo):.1%}); "
      f"de ellos, efectos individuales de jugador: {es_efecto_jugador.sum()}")
print(f"ess_bulk minimo entre todos los parametros: {resumen_completo['ess_bulk'].min():.0f}")
print(problematicos[["mean", "sd", "ess_bulk", "r_hat"]].to_string())

parametros totales: 540
r_hat maximo: 1.0147
parametros con r_hat > 1.01: 16 (3.0%); de ellos, efectos individuales de jugador: 16
ess_bulk minimo entre todos los parametros: 543
                           mean    sd ess_bulk r_hat
1|player_id[00-0023496]  -0.074  0.28     6175  1.01
1|player_id[00-0029336]  -0.023  0.26     6135  1.01
1|player_id[00-0030091]   0.072  0.26     6571  1.01
1|player_id[00-0030669]   0.097  0.22     5447  1.01
1|player_id[00-0031545]  -0.189  0.26     5455  1.01
1|player_id[00-0031600]  -0.048  0.29     5430  1.01
1|player_id[00-0032385]  -0.014  0.18     5623  1.01
1|player_id[00-0032553]  -0.011  0.28     5507  1.01
1|player_id[00-0032795]  -0.162  0.26     5522  1.01
1|player_id[00-0032801]  -0.015  0.27     6091  1.01
1|player_id[00-0033379]  -0.019  0.28     4403  1.01
1|player_id[00-0034411]   0.012  0.21     4054  1.01
1|player_id[00-0035350]   0.176  0.25     4459  1.01
1|player_id[00-0036110]  -0.049  0.27     5146  1.01
1|player_id[00-0036326]   

**Lectura.** De 540 parámetros, 16 (3.0%) tienen `r_hat` entre 1.01 y 1.0147, y los dieciséis son efectos de
jugadores individuales; ningún coeficiente fijo está en esa situación. El tamaño de muestra
efectivo más bajo es 543. Es una convergencia aceptable para comparar predicciones; para leer el
efecto de esos dieciséis jugadores en particular convendría muestrear más.

## 4. Los coeficientes se leen directo

Media posterior e intervalo de credibilidad al 89% (el de `arviz` por defecto). Si el intervalo no
cruza cero, el efecto es claro con estos datos.

In [6]:
coeficientes = az.summary(idata, var_names=["Intercept"] + features_ridge, filter_vars=None)
coeficientes = coeficientes.reindex(coeficientes["mean"].abs().sort_values(ascending=False).index)
coeficientes["credible"] = np.where(
    (coeficientes["eti89_lb"] > 0) | (coeficientes["eti89_ub"] < 0), "sí", "no (cruza cero)"
)
print("top 10 coeficientes por magnitud, con si su intervalo de credibilidad cruza cero:")
print(coeficientes[["mean", "eti89_lb", "eti89_ub", "credible"]].head(10).to_string())

top 10 coeficientes por magnitud, con si su intervalo de credibilidad cruza cero:
                                     mean eti89_lb eti89_ub         credible
wopr_last3_avg                       0.63    -0.13      1.4  no (cruza cero)
catch_rate_career_avg              -0.518    -0.87    -0.16               sí
depth_team                           -0.3    -0.38    -0.23               sí
Intercept                           -0.29     -1.4     0.89  no (cruza cero)
anios_experiencia_bucket[Rookie]   -0.095    -0.26    0.059  no (cruza cero)
receptions_season_avg               0.095   -0.029     0.21  no (cruza cero)
receptions_career_avg               0.088   -0.018      0.2  no (cruza cero)
receiving_20_last3_avg             -0.087     -0.2    0.025  no (cruza cero)
wopr_last5_avg                      -0.07    -0.99     0.85  no (cruza cero)
anios_experiencia_bucket[6+ años]   0.057    -0.19     0.31  no (cruza cero)


**Lectura.** Dos de los diez coeficientes tienen un intervalo que no cruza cero. `depth_team` (−0.30): cada
lugar más abajo en la alineación multiplica la tasa esperada de touchdowns por exp(−0.30) = 0.74,
es decir, unos 26% menos. `catch_rate_career_avg` (−0.518 por unidad de proporción): con lo demás
igual, los receptores que atrapan una proporción mayor de sus envíos anotan menos. Una explicación
posible, que aquí no se prueba, es que un porcentaje alto de atrapadas va con rutas cortas y uno
bajo con envíos profundos.

Dos cuidados al leer esta tabla. Las variables no están estandarizadas, así que ordenar por
magnitud mezcla escalas (`catch_rate_career_avg` es una proporción entre 0 y 1 y `draft_pick` cuenta lugares del draft). Y ocho de los diez
coeficientes cruzan cero, `wopr_last3_avg` incluido: con 30 variables correlacionadas entre sí, el
efecto individual de cada una queda mal identificado aunque la predicción conjunta funcione.

## 5. El efecto individual de un jugador real

El efecto por jugador (`1|player_id`) es lo que ningún modelo anterior tenía: lo que un receptor
anota de más o de menos respecto a otro con sus mismas variables. Ja'Marr Chase como ejemplo:

In [7]:
chase_id = data.cargar_jugadores().pipe(lambda j: j[j["display_name"] == "Ja'Marr Chase"])["gsis_id"].iloc[0]
efectos = idata.posterior["1|player_id"].mean(dim=["chain", "draw"]).to_dataframe().reset_index()
efectos.columns = ["player_id", "efecto_log"]

efecto_chase = efectos.loc[efectos["player_id"] == chase_id, "efecto_log"].iloc[0]
cambio = np.exp(efecto_chase) - 1
print(f"efecto individual de Ja'Marr Chase (escala log): {efecto_chase:.3f}")
print(f"en escala de tasa: exp({efecto_chase:.3f}) = {np.exp(efecto_chase):.2f}x, es decir, una tasa esperada de TDs "
      f"{abs(cambio):.0%} {'mas alta' if cambio > 0 else 'mas baja'} que la de un receptor con sus mismas variables")
print(f"\nrango de efectos en los {len(efectos)} jugadores de entrenamiento: "
      f"[{efectos['efecto_log'].min():.3f}, {efectos['efecto_log'].max():.3f}], mediana={efectos['efecto_log'].median():.3f}")

efecto individual de Ja'Marr Chase (escala log): 0.253
en escala de tasa: exp(0.253) = 1.29x, es decir, una tasa esperada de TDs 29% mas alta que la de un receptor con sus mismas variables

rango de efectos en los 505 jugadores de entrenamiento: [-0.372, 0.545], mediana=-0.018


In [8]:
semanas_chase = wr.loc[train_mask & (wr["player_id"] == chase_id)].groupby("season").size()
print("semanas de Ja'Marr Chase en entrenamiento, por temporada:", semanas_chase.to_dict())

semanas de Ja'Marr Chase en entrenamiento, por temporada: {2021: 17}


**Lectura.** El efecto de Chase es +0.253 en escala logarítmica: su tasa esperada de touchdowns es
29% mayor que la de un receptor con sus mismas variables. Ese efecto sale de una sola temporada: en
entrenamiento Chase solo aparece en 2021 (17 semanas).

Entre los 505 receptores de entrenamiento los efectos van de −0.372 a +0.545, con mediana −0.018.
Una mediana cercana a cero es lo que se espera del encogimiento: sin mucha historia, el efecto de un
receptor se queda cerca del promedio de la posición.

## 6. Comparación contra el modelo elegido

Se predice el valor esperado de la Binomial Negativa (`kind="response_params"`, la variable `mu`)
para validación y prueba. Los jugadores que no estaban en entrenamiento reciben efectos
muestreados de la distribución de la población, así que su predicción varía un poco de una corrida
a otra. Se compara contra el XGBoost Poisson elegido y contra el modelo de dos etapas de
[`4.3_modelos_touchdowns.ipynb`](4.3_modelos_touchdowns.ipynb).

In [9]:
def datos_para(mascara):
    datos = imputar(wr.loc[mascara, features_ridge], ajustar=False)
    datos["player_id"] = wr.loc[mascara, "player_id"].astype(str).to_numpy()
    return datos

def predecir_jerarquico(datos):
    p = modelo.predict(idata, data=datos, kind="response_params", inplace=False, sample_new_groups=True)
    return p.posterior["mu"].mean(dim=["chain", "draw"]).to_numpy()

datos_val, datos_test = datos_para(val_mask), datos_para(test_mask)
pred_jerarquico = pd.Series(np.nan, index=wr.index)
pred_jerarquico[val_mask] = predecir_jerarquico(datos_val)
pred_jerarquico[test_mask] = predecir_jerarquico(datos_test)

features_arbol = features.columnas_por_modelo("arbol")
xgb_elegido = XGBRegressor(**CONFIG_ELEGIDA[target], random_state=0, n_jobs=-1).fit(
    wr.loc[train_mask, features_arbol], wr.loc[train_mask, target])
pred_elegido = pd.Series(np.clip(xgb_elegido.predict(wr[features_arbol]), 0, None), index=wr.index)
pred_hurdle = experimentos.entrenar_hurdle(wr, target, features_arbol, train_mask, val_mask, test_mask)["pred"].clip(lower=0)

media = wr.loc[train_mask, target].mean()
y = wr[target]
filas = {}
for nombre, pred in (("XGBoost Poisson elegido", pred_elegido), ("Hurdle", pred_hurdle), ("Jerárquico + Binomial Negativa", pred_jerarquico)):
    for parte, mascara in (("val", val_mask), ("test", test_mask)):
        m = modeling.metricas_target(target, y[mascara], pred[mascara], media)
        for k in ("deviance_poisson", "d2_oos", "r2_oos", "sesgo", "pendiente_calibracion", "auc_anota", "mae"):
            filas.setdefault(nombre, {})[f"{parte}_{k}"] = m[k]
comparacion = pd.DataFrame(filas).T
c = experimentos.diferencia_bootstrap(wr[val_mask], y[val_mask], pred_jerarquico[val_mask], pred_elegido[val_mask], "deviance_poisson")
print(f"jerarquico menos XGBoost elegido, deviance de validacion = {c['diferencia']:+.4f} "
      f"(IC 95%: {c['ic_inferior']:+.4f} a {c['ic_superior']:+.4f}) -> "
      f"{'empate' if c['empate'] else ('mejora real' if c['diferencia'] < 0 else 'empeora')}")
comparacion.round(4)

/tmp/ipykernel_7/2254382861.py:7: FutureWarning: 'sample_new_groups' is deprecated and will be removed in a future version. New groups will be handled automatically.
  p = modelo.predict(idata, data=datos, kind="response_params", inplace=False, sample_new_groups=True)


/tmp/ipykernel_7/2254382861.py:7: FutureWarning: 'sample_new_groups' is deprecated and will be removed in a future version. New groups will be handled automatically.
  p = modelo.predict(idata, data=datos, kind="response_params", inplace=False, sample_new_groups=True)


jerarquico menos XGBoost elegido, deviance de validacion = +0.0129 (IC 95%: +0.0037 a +0.0212) -> empeora


,val_deviance_poisson,val_d2_oos,val_r2_oos,val_sesgo,val_pendiente_calibracion,val_auc_anota,val_mae,test_deviance_poisson,test_d2_oos,test_r2_oos,test_sesgo,test_pendiente_calibracion,test_auc_anota,test_mae
XGBoost Poisson elegido,0.6167,0.1223,0.0826,0.0261,0.9604,0.7129,0.3081,0.6252,0.1355,0.0877,0.0001,1.0104,0.7201,0.3045
Hurdle,0.6228,0.1135,0.0745,0.0290,0.8873,0.7057,0.3102,0.6356,0.1212,0.0751,0.0021,0.9232,0.7068,0.3076
Jerárquico + Binomial Negativa,0.6296,0.1039,0.0675,0.0368,0.8021,0.7040,0.3128,0.6389,0.1166,0.0745,-0.0058,0.8393,0.7086,0.2993


**Lectura.** El modelo jerárquico queda por detrás del XGBoost Poisson elegido. En validación su deviance es
0.6296 contra 0.6167: una diferencia de +0.0129 con intervalo de +0.0037 a +0.0212, que no incluye
cero, así que es peor de forma real y no un empate. También está peor calibrado (pendiente de 0.802,
fuera del rango 0.9-1.1 que piden los requisitos) y explica menos deviance (D² de 0.104 contra
0.122). La prueba confirma el orden: 0.6389 contra 0.6252.

La tabla muestra de paso por qué no se elige con MAE: en la prueba el jerárquico tiene el MAE más
bajo de los tres (0.2993 contra 0.3045), con peor deviance, peor D² y peor calibración. Su sesgo
negativo en la prueba (−0.006) lo acerca a la mediana, que es lo que premia el MAE.

## 7. ¿El efecto por jugador ayuda más a los receptores conocidos?

La hipótesis natural es que el modelo jerárquico funciona mejor con jugadores que ya vio en
entrenamiento (tienen un efecto propio estimado) que con jugadores nuevos (reciben el de la
población). Se revisa en la prueba, separando los dos grupos.

In [10]:
jugadores_train = set(datos_train["player_id"].unique())
visto = wr.loc[test_mask, "player_id"].astype(str).isin(jugadores_train)
filas = []
for grupo, mascara_grupo in (("jugador nuevo (no en entrenamiento)", ~visto), ("ya visto en entrenamiento", visto)):
    indices = wr.index[test_mask][mascara_grupo.to_numpy()]
    for nombre, pred in (("jerarquico", pred_jerarquico), ("XGBoost elegido", pred_elegido)):
        m = modeling.metricas_target(target, y[indices], pred[indices], media)
        filas.append({"grupo": grupo, "modelo": nombre, "n": len(indices), "deviance_poisson": m["deviance_poisson"], "d2_oos": m["d2_oos"]})
pd.DataFrame(filas).round(4)

,grupo,modelo,n,deviance_poisson,d2_oos
0,jugador nuevo (no en entrenamiento),jerarquico,2702,0.6299,0.1020
1,jugador nuevo (no en entrenamiento),XGBoost elegido,2702,0.6107,0.1294
2,ya visto en entrenamiento,jerarquico,2250,0.6498,0.1330
3,ya visto en entrenamiento,XGBoost elegido,2250,0.6427,0.1424


**Lectura.** La hipótesis se cumple en parte. Con jugadores nuevos (2,702 filas de prueba), el jerárquico queda
lejos del XGBoost: deviance de 0.6299 contra 0.6107. Con jugadores ya vistos (2,250 filas), la
distancia se reduce a 0.6498 contra 0.6427. El efecto propio acorta la distancia donde existe, pero
ni en ese grupo alcanza al XGBoost.

## 8. ¿Ayuda mezclarlo con el XGBoost elegido?

Aunque por sí solo sea peor, el jerárquico usa información que el XGBoost no tiene: un efecto propio por
jugador. Si sus errores no coinciden con los del XGBoost, una mezcla `w · XGBoost + (1 − w) ·
jerárquico` podría mejorar a los dos. El peso se elige con la deviance de validación, y la mejor
mezcla se compara contra el XGBoost solo con la regla de 4.1, también con el intervalo ajustado por
los 7 pesos probados.

In [11]:
print(f"correlacion de predicciones en validacion, XGBoost elegido vs jerarquico: "
      f"{np.corrcoef(pred_elegido[val_mask], pred_jerarquico[val_mask])[0, 1]:.3f}")
PESOS = (1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.3, 0.0)
filas, mezclas = {}, {}
for w in PESOS:
    pred = w * pred_elegido + (1 - w) * pred_jerarquico
    mezclas[w] = pred
    val = modeling.metricas_target(target, y[val_mask], pred[val_mask], media)
    test = modeling.metricas_target(target, y[test_mask], pred[test_mask], media)
    filas[f"w = {w}"] = {"val_deviance": val["deviance_poisson"], "val_d2_oos": val["d2_oos"], "val_r2_oos": val["r2_oos"],
                         "val_pendiente": val["pendiente_calibracion"], "test_deviance": test["deviance_poisson"],
                         "test_d2_oos": test["d2_oos"], "test_r2_oos": test["r2_oos"]}
tabla_mezcla = pd.DataFrame(filas).T
mejor_w = PESOS[int(np.argmin(tabla_mezcla["val_deviance"].to_numpy()))]
print(f"mejor peso en validacion: w = {mejor_w}")
if mejor_w < 1:
    for nivel in (0.95, 1 - 0.05 / (len(PESOS) - 1)):
        c_mezcla = experimentos.diferencia_bootstrap(wr[val_mask], y[val_mask], mezclas[mejor_w][val_mask], pred_elegido[val_mask],
                                                     "deviance_poisson", n_remuestreos=20000, nivel=nivel)
        print(f"  contra el XGBoost solo, intervalo de {nivel:.1%}: {c_mezcla['diferencia']:+.4f} "
              f"({c_mezcla['ic_inferior']:+.4f} a {c_mezcla['ic_superior']:+.4f}) -> "
              f"{'empate' if c_mezcla['empate'] else 'diferencia real'}")
print()
print(tabla_mezcla.round(4).to_string())

correlacion de predicciones en validacion, XGBoost elegido vs jerarquico: 0.886
mejor peso en validacion: w = 0.9
  contra el XGBoost solo, intervalo de 95.0%: -0.0003 (-0.0013 a +0.0007) -> empate
  contra el XGBoost solo, intervalo de 99.3%: -0.0003 (-0.0017 a +0.0011) -> empate

         val_deviance  val_d2_oos  val_r2_oos  val_pendiente  test_deviance  test_d2_oos  test_r2_oos
w = 1.0        0.6167      0.1223      0.0826         0.9604         0.6252       0.1355       0.0877
w = 0.9        0.6164      0.1227      0.0835         0.9640         0.6246       0.1364       0.0886
w = 0.8        0.6165      0.1225      0.0838         0.9623         0.6244       0.1367       0.0890
w = 0.7        0.6170      0.1217      0.0835         0.9553         0.6247       0.1363       0.0890
w = 0.6        0.6179      0.1206      0.0828         0.9435         0.6253       0.1354       0.0884
w = 0.5        0.6190      0.1189      0.0816         0.9274         0.6264       0.1339       0.0873
w =

**Lectura.** Las predicciones de los dos modelos se parecen menos entre sí (correlación de 0.886 en validación)
que las de los modelos de árboles en
[`4.6_ensamble_simple.ipynb`](4.6_ensamble_simple.ipynb), así que había margen para que la mezcla
ayudara. El mejor peso es w = 0.9 (90% XGBoost, 10% jerárquico), con una deviance de validación de
0.6164 contra 0.6167 del XGBoost solo. La diferencia, −0.0003, tiene un intervalo de −0.0013 a
+0.0007 que incluye el cero aun sin ajustar (ajustado por los 7 pesos: −0.0017 a +0.0011): es un
empate. En prueba la mezcla queda también apenas
por debajo (0.6246 contra 0.6252; D² de 0.1364 contra 0.1355), una diferencia del mismo orden que la
de validación, que ya fue empate.

Una diferencia así no justifica sumar un modelo bayesiano al flujo semanal, que requeriría cargar
`bambi` y promediar 2,000 muestras por receptor cada semana.

## 9. Guardar el modelo para 4.9

El modelo no se adopta, pero
[`4.9_ventana_entrenamiento.ipynb`](4.9_ventana_entrenamiento.ipynb) lo reutiliza, con sus
imputadores, para no repetir el ajuste (585 segundos en esta corrida). Por eso se guarda en
`weekly/wr/models/exploratorio/` solo como insumo local: pesa varios MB, se regenera corriendo este
notebook y no se versiona; su metadata sí.

`idata.to_netcdf()`, el formato estándar de `arviz`, guarda sin error pero falla al recargar con las
versiones instaladas de `h5netcdf`/`xarray` (ver [`docs/HALLAZGOS.md`](../../../docs/HALLAZGOS.md)),
así que se guarda con `joblib` y se verifica la recarga.

In [12]:
import json, os
import joblib

os.makedirs("../models/exploratorio", exist_ok=True)
joblib.dump({
    "idata": idata,
    "formula": formula,
    "family": "negativebinomial",
    "imputador_num": imputador_num,
    "imputador_cat": imputador_cat,
    "cols_numericas": cols_numericas,
    "features_ridge": features_ridge,
}, "../models/exploratorio/receiving_tds_jerarquico.joblib")

# verificar que recargar y predecir con el objeto guardado da el mismo resultado
paquete_recargado = joblib.load("../models/exploratorio/receiving_tds_jerarquico.joblib")
modelo_verificacion = bmb.Model(paquete_recargado["formula"], datos_train, family=paquete_recargado["family"])
pred_verificacion = modelo_verificacion.predict(
    paquete_recargado["idata"], data=datos_test, kind="response_params", inplace=False, sample_new_groups=True
)
pred_media_verificacion = pred_verificacion.posterior["mu"].mean(dim=["chain", "draw"]).to_numpy()
m_verificacion = modeling.metricas_target(target, y[test_mask], pred_media_verificacion, media)
print(f"deviance de prueba con el modelo recargado desde disco: {m_verificacion['deviance_poisson']:.4f}")
print(f"deviance de prueba antes de guardar:              {comparacion.loc['Jerárquico + Binomial Negativa', 'test_deviance_poisson']:.4f}")

metadata = {
    "target": target,
    "modelo": "bambi.Model, familia negativebinomial, efecto aleatorio (1|player_id)",
    "se_adopta": bool((not c["empate"]) and c["diferencia"] < 0),
    "formula": formula,
    "n_efectos_jugador": int(datos_train["player_id"].nunique()),
    "diagnostico_convergencia": {
        "r_hat_max": round(float(resumen_completo["r_hat"].max()), 4),
        "parametros_r_hat_mayor_1_01": int(len(problematicos)),
        "parametros_totales": int(len(resumen_completo)),
    },
    "formato_guardado": "joblib (el netCDF de arviz falla al recargar con las versiones instaladas de h5netcdf/xarray)",
    "metricas_validacion_2022_2023": {k.replace("val_", ""): round(float(v), 4) for k, v in comparacion.loc["Jerárquico + Binomial Negativa"].items() if k.startswith("val_")},
    "metricas_prueba_2024_2025": {k.replace("test_", ""): round(float(v), 4) for k, v in comparacion.loc["Jerárquico + Binomial Negativa"].items() if k.startswith("test_")},
    "fecha_de_guardado": pd.Timestamp.now(tz="UTC").isoformat(),
    "limitaciones_conocidas": [
        f"jugadores de prueba sin efecto propio (no estaban en entrenamiento): {int((~visto).sum())} de {len(visto)} filas; reciben el efecto de la poblacion",
        f"{len(problematicos)} de {len(resumen_completo)} parametros con r_hat > 1.01 ({int(es_efecto_jugador.sum())} son efectos individuales de jugador)",
    ],
}
with open("../models/exploratorio/receiving_tds_jerarquico_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("\nguardado en models/exploratorio: receiving_tds_jerarquico.joblib + metadata")

/tmp/ipykernel_7/634217359.py:18: FutureWarning: 'sample_new_groups' is deprecated and will be removed in a future version. New groups will be handled automatically.
  pred_verificacion = modelo_verificacion.predict(


deviance de prueba con el modelo recargado desde disco: 0.6393
deviance de prueba antes de guardar:              0.6389

guardado en models/exploratorio: receiving_tds_jerarquico.joblib + metadata


## Cierre

El modelo jerárquico con Binomial Negativa **no se adopta** para touchdowns: en validación es peor
que el XGBoost Poisson elegido con una diferencia que no incluye cero (deviance +0.0129, IC 95%:
+0.0037 a +0.0212) y no cumple el requisito de calibración (pendiente 0.802). Tampoco aporta
mezclado con el XGBoost: el mejor peso (90% XGBoost, 10% jerárquico) empata con el XGBoost solo
(−0.0003 de deviance, IC 95%: −0.0013 a +0.0007). Lo que sí deja es
una lectura explicable del problema: coeficientes con intervalos de credibilidad (`depth_team` resta
unos 26% de la tasa por cada lugar en la alineación) y un efecto legible por jugador (Chase, +29%,
estimado con una sola temporada). El modelo se guarda en `weekly/wr/models/exploratorio/` solo como
insumo de 4.9, fuera del control de versiones.

La recarga desde disco da una deviance de 0.6393 contra 0.6389 antes de guardar. La diferencia
viene de que los jugadores nuevos reciben un efecto muestreado de la población en cada predicción,
no de un error al guardar.

Anterior: [`4.6_ensamble_simple.ipynb`](4.6_ensamble_simple.ipynb) · Siguiente:
[`4.8_jerarquico_recepciones_yardas.ipynb`](4.8_jerarquico_recepciones_yardas.ipynb)